## Pre-trained MobileNet

In [ ]:
# ============================================================
# CELL 1
# IMPORTS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm import tqdm

from numpy.linalg import inv

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.linear_model import (
    RidgeClassifier,
    LogisticRegression
)

from sklearn.preprocessing import LabelEncoder

Mounted at /content/drive


In [ ]:
# ============================================================
# CELL 2
# DATASET SETUP + ANALYSIS
# ============================================================

DATASET_ROOT = (
    "/content/drive/MyDrive/TinyBayes/amini_dataset"
)

BASE_DIR = os.path.join(
    DATASET_ROOT,
    "dataset"
)

CSV_PATH_TRAIN = os.path.join(
    DATASET_ROOT,
    "Train.csv"
)

CSV_PATH_TEST = os.path.join(
    DATASET_ROOT,
    "Test.csv"
)

IMAGE_DIR_TRAIN = os.path.join(
    BASE_DIR,
    "images",
    "train"
)

IMAGE_DIR_TEST = os.path.join(
    BASE_DIR,
    "images",
    "test"
)

IMAGE_DIR_VAL = os.path.join(
    BASE_DIR,
    "images",
    "val"
)

LABEL_DIR_TRAIN = os.path.join(
    BASE_DIR,
    "labels",
    "train"
)

SAVE_DIR = (
    "/content/drive/MyDrive/TinyBayes/run_2000"
)

os.makedirs(
    IMAGE_DIR_VAL,
    exist_ok=True
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# Verify dataset
# ------------------------------------------------------------

print("=" * 60)
print("VERIFYING DATASET")
print("=" * 60)

print(
    "Train CSV      :",
    os.path.exists(CSV_PATH_TRAIN)
)

print(
    "Test CSV       :",
    os.path.exists(CSV_PATH_TEST)
)

print(
    "Train Images   :",
    os.path.exists(IMAGE_DIR_TRAIN)
)

print(
    "Test Images    :",
    os.path.exists(IMAGE_DIR_TEST)
)

print(
    "Validation Dir :",
    os.path.exists(IMAGE_DIR_VAL)
)

print(
    "Train Labels   :",
    os.path.exists(LABEL_DIR_TRAIN)
)

print(
    "Output Folder  :",
    os.path.exists(SAVE_DIR)
)

# ------------------------------------------------------------
# Load CSVs
# ------------------------------------------------------------

df_train = pd.read_csv(
    CSV_PATH_TRAIN
)

df_test = pd.read_csv(
    CSV_PATH_TEST
)

# ------------------------------------------------------------
# Dataset Summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

print(
    "Train Samples:",
    len(df_train)
)

print(
    "Test Samples:",
    len(df_test)
)

print(
    "\nTrain Shape:",
    df_train.shape
)

print(
    "Test Shape:",
    df_test.shape
)

print(
    "\nTrain Columns:"
)

print(
    df_train.columns.tolist()
)

print(
    "\nClass Distribution:"
)

print(
    df_train["class"].value_counts()
)

print(
    "\nUnique Classes:"
)

print(
    df_train["class"].unique()
)

print(
    "\nUnique Images:"
)

print(
    df_train["Image_ID"].nunique()
)

print(
    "\nFirst 5 Rows:"
)

display(
    df_train.head()
)

# ------------------------------------------------------------
# Label file check
# ------------------------------------------------------------

label_files = os.listdir(
    LABEL_DIR_TRAIN
)

print(
    "\nNumber of Label Files:",
    len(label_files)
)

print(
    "\nExample Label File:"
)

print(
    label_files[0]
)

sample_label = os.path.join(
    LABEL_DIR_TRAIN,
    label_files[0]
)

with open(sample_label) as f:

    print(
        "\nContents:"
    )

    print(
        f.read()
    )


VERIFYING DATASET
Train CSV      : True
Test CSV       : True
Train Images   : True
Test Images    : True
Validation Dir : True
Train Labels   : True
Output Folder  : True

DATASET SUMMARY
Train Samples: 9792
Test Samples: 1626

Train Shape: (9792, 9)
Test Shape: (1626, 9)

Train Columns:
['Image_ID', 'class', 'confidence', 'ymin', 'xmin', 'ymax', 'xmax', 'class_id', 'ImagePath']

Class Distribution:
class
healthy        4280
cssvd          3241
anthracnose    2271
Name: count, dtype: int64

Unique Classes:
['healthy' 'anthracnose' 'cssvd']

Unique Images:
5529

First 5 Rows:


,Image_ID,class,confidence,ymin,xmin,ymax,xmax,class_id,ImagePath
0,ID_nBgcAR.jpg,healthy,1.0,75.0,15.0,162.0,195.0,2,dataset/images/train/ID_nBgcAR.jpg
1,ID_nBgcAR.jpg,healthy,1.0,58.0,1.0,133.0,171.0,2,dataset/images/train/ID_nBgcAR.jpg
2,ID_nBgcAR.jpg,healthy,1.0,42.0,29.0,377.0,349.0,2,dataset/images/train/ID_nBgcAR.jpg
3,ID_Kw2v8A.jpg,healthy,1.0,112.0,124.0,404.0,341.0,2,dataset/images/train/ID_Kw2v8A.jpg
4,ID_Kw2v8A.jpg,healthy,1.0,148.0,259.0,413.0,412.0,2,dataset/images/train/ID_Kw2v8A.jpg



Number of Label Files: 5529

Example Label File:
ID_r8Etsf.txt

Contents:
2 0.458854 0.583203 0.271875 0.553906
2 0.827604 0.421484 0.344792 0.197656



In [ ]:
import os

train_images = [
    f for f in os.listdir(IMAGE_DIR_TRAIN)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print("Images in train folder:", len(train_images))

Images in train folder: 5529


In [ ]:
# ============================================================
# CREATE CLEAN IMAGE-LEVEL DATASET
# FOR ANDROID APP PIPELINE
# ============================================================

TRAIN_SIZE = 2000
VAL_SIZE = 500

# ------------------------------------------------------------
# Remove images that belong to multiple classes
# ------------------------------------------------------------

image_class_count = (
    df_train
    .groupby("Image_ID")["class"]
    .nunique()
)

ambiguous_images = set(
    image_class_count[
        image_class_count > 1
    ].index
)

print(
    "Images with multiple classes:",
    len(ambiguous_images)
)

# ------------------------------------------------------------
# Keep only clean images
# ------------------------------------------------------------

clean_df = df_train[
    ~df_train["Image_ID"].isin(
        ambiguous_images
    )
]

# ------------------------------------------------------------
# One row per image
# ------------------------------------------------------------

unique_images = (
    clean_df[
        ["Image_ID", "class"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "Unique clean images:",
    len(unique_images)
)

# ------------------------------------------------------------
# Class distribution
# ------------------------------------------------------------

print("\nClass Distribution:")

print(
    unique_images["class"]
    .value_counts()
)

# ============================================================
# CREATE BALANCED TRAINING SET
# ============================================================

train_per_class = TRAIN_SIZE // 3

train_parts = []

for cls in [
    "healthy",
    "cssvd",
    "anthracnose"
]:

    cls_df = unique_images[
        unique_images["class"] == cls
    ]

    sampled = cls_df.sample(
        n=train_per_class,
        random_state=42
    )

    train_parts.append(
        sampled
    )

df_train_balanced = pd.concat(
    train_parts
).reset_index(
    drop=True
)

# ------------------------------------------------------------
# Add remainder if needed
# ------------------------------------------------------------

remaining_needed = (
    TRAIN_SIZE -
    len(df_train_balanced)
)

if remaining_needed > 0:

    remaining_pool = unique_images[
        ~unique_images["Image_ID"].isin(
            df_train_balanced["Image_ID"]
        )
    ]

    extra = remaining_pool.sample(
        n=remaining_needed,
        random_state=42
    )

    df_train_balanced = pd.concat([
        df_train_balanced,
        extra
    ]).reset_index(
        drop=True
    )

# ============================================================
# CREATE BALANCED VALIDATION SET
# ============================================================

remaining_images = unique_images[
    ~unique_images["Image_ID"].isin(
        df_train_balanced["Image_ID"]
    )
]

val_per_class = VAL_SIZE // 3

val_parts = []

for cls in [
    "healthy",
    "cssvd",
    "anthracnose"
]:

    cls_df = remaining_images[
        remaining_images["class"] == cls
    ]

    sampled = cls_df.sample(
        n=val_per_class,
        random_state=44
    )

    val_parts.append(
        sampled
    )

validation_data = pd.concat(
    val_parts
).reset_index(
    drop=True
)

remaining_needed_val = (
    VAL_SIZE -
    len(validation_data)
)

if remaining_needed_val > 0:

    remaining_pool = remaining_images[
        ~remaining_images["Image_ID"].isin(
            validation_data["Image_ID"]
        )
    ]

    extra = remaining_pool.sample(
        n=remaining_needed_val,
        random_state=44
    )

    validation_data = pd.concat([
        validation_data,
        extra
    ]).reset_index(
        drop=True
    )

# ============================================================
# VERIFY NO LEAKAGE
# ============================================================

overlap = set(
    df_train_balanced["Image_ID"]
).intersection(
    set(
        validation_data["Image_ID"]
    )
)

print("\n")
print("=" * 60)
print("LEAKAGE CHECK")
print("=" * 60)

print(
    "Overlapping images:",
    len(overlap)
)

# ============================================================
# SUMMARY
# ============================================================

print("\n")
print("=" * 60)
print("TRAIN SET")
print("=" * 60)

print(
    df_train_balanced["class"]
    .value_counts()
)

print(
    "Total:",
    len(df_train_balanced)
)

print("\n")
print("=" * 60)
print("VALIDATION SET")
print("=" * 60)

print(
    validation_data["class"]
    .value_counts()
)

print(
    "Total:",
    len(validation_data)
)

# ============================================================
# SAVE SPLITS
# ============================================================

TRAIN_SPLIT_PATH = os.path.join(
    SAVE_DIR,
    "train_2000_unique_images.csv"
)

VAL_SPLIT_PATH = os.path.join(
    SAVE_DIR,
    "val_500_unique_images.csv"
)

df_train_balanced.to_csv(
    TRAIN_SPLIT_PATH,
    index=False
)

validation_data.to_csv(
    VAL_SPLIT_PATH,
    index=False
)

print("\nSaved:")

print(TRAIN_SPLIT_PATH)
print(VAL_SPLIT_PATH)

Images with multiple classes: 6
Unique clean images: 5523

Class Distribution:
class
cssvd          2237
healthy        1720
anthracnose    1566
Name: count, dtype: int64


LEAKAGE CHECK
Overlapping images: 0


TRAIN SET
class
healthy        667
cssvd          667
anthracnose    666
Name: count, dtype: int64
Total: 2000


VALIDATION SET
class
healthy        167
cssvd          167
anthracnose    166
Name: count, dtype: int64
Total: 500

Saved:
/content/drive/MyDrive/TinyBayes/run_2000/train_2000_unique_images.csv
/content/drive/MyDrive/TinyBayes/run_2000/val_500_unique_images.csv


In [ ]:
# ============================================================
# STEP 4
# EXTRACT FULL-IMAGE TFLITE MOBILENET FEATURES
# ============================================================

import tensorflow as tf
import pandas as pd
import numpy as np
import cv2
import os

from tqdm import tqdm

MODEL_PATH = (
    "/content/drive/MyDrive/TinyBayes/run_2000/"
    "mobilenet_v3_small.tflite"
)

# ------------------------------------------------------------
# Load TFLite MobileNet
# ------------------------------------------------------------

interpreter = tf.lite.Interpreter(
    model_path=MODEL_PATH
)

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# ------------------------------------------------------------
# Feature extraction function
# ------------------------------------------------------------

def extract_full_image_features(
    dataframe,
    image_dir,
    training=True
):

    extracted_features = []

    for _, row in tqdm(
        dataframe.iterrows(),
        total=len(dataframe)
    ):

        image_path = os.path.join(
            image_dir,
            row["Image_ID"]
        )

        image = cv2.imread(image_path)

        if image is None:
            continue

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        # ----------------------------------------------------
        # EXACTLY SAME AS ANDROID APP
        # ----------------------------------------------------

        image = cv2.resize(
            image,
            (224, 224)
        ).astype(np.float32)

        image /= 255.0

        image[:, :, 0] = (
            image[:, :, 0] - 0.485
        ) / 0.229

        image[:, :, 1] = (
            image[:, :, 1] - 0.456
        ) / 0.224

        image[:, :, 2] = (
            image[:, :, 2] - 0.406
        ) / 0.225

        input_tensor = np.expand_dims(
            image,
            axis=0
        ).astype(np.float32)

        interpreter.set_tensor(
            input_details[0]["index"],
            input_tensor
        )

        interpreter.invoke()

        feat = interpreter.get_tensor(
            output_details[0]["index"]
        )[0]

        if training:

            extracted_features.append([
                *feat,
                row["class"]
            ])

        else:

            extracted_features.append([
                row["Image_ID"],
                row["class"],
                *feat
            ])

    feature_columns = [
        f"feat_{i}"
        for i in range(576)
    ]

    if training:

        feature_columns.append(
            "label"
        )

    else:

        feature_columns = (
            ["Image_ID", "class"]
            + feature_columns
        )

    return pd.DataFrame(
        extracted_features,
        columns=feature_columns
    )

# ------------------------------------------------------------
# Extract Training Features
# ------------------------------------------------------------

print(
    "Extracting training features..."
)

extracted_features_train = (
    extract_full_image_features(
        df_train_balanced,
        IMAGE_DIR_TRAIN,
        training=True
    )
)

print(
    "Training shape:",
    extracted_features_train.shape
)

# ------------------------------------------------------------
# Extract Validation Features
# ------------------------------------------------------------

print(
    "\nExtracting validation features..."
)

extracted_features_val = (
    extract_full_image_features(
        validation_data,
        IMAGE_DIR_TRAIN,
        training=False
    )
)

print(
    "Validation shape:",
    extracted_features_val.shape
)

# ------------------------------------------------------------
# Save Features
# ------------------------------------------------------------

TRAIN_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "full_image_tflite_train_features.csv"
)

VAL_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "full_image_tflite_val_features.csv"
)

extracted_features_train.to_csv(
    TRAIN_FEATURE_PATH,
    index=False
)

extracted_features_val.to_csv(
    VAL_FEATURE_PATH,
    index=False
)

print("\nSaved:")

print(TRAIN_FEATURE_PATH)
print(VAL_FEATURE_PATH)

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Extracting training features...


100%|██████████| 2000/2000 [03:39<00:00,  9.11it/s]


Training shape: (2000, 577)

Extracting validation features...


100%|██████████| 500/500 [12:49<00:00,  1.54s/it]


Validation shape: (500, 578)

Saved:
/content/drive/MyDrive/TinyBayes/run_2000/full_image_tflite_train_features.csv
/content/drive/MyDrive/TinyBayes/run_2000/full_image_tflite_val_features.csv


In [ ]:
# ============================================================
# STEP 4
# EXTRACT FULL-IMAGE TFLITE MOBILENET FEATURES
# ============================================================

import tensorflow as tf
import pandas as pd
import numpy as np
import cv2
import os

from tqdm import tqdm

TRAIN_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "full_image_tflite_train_features.csv"
)

VAL_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "full_image_tflite_val_features.csv"
)

# ------------------------------------------------------------
# Load Existing Features If Available
# ------------------------------------------------------------

if (
    os.path.exists(TRAIN_FEATURE_PATH)
    and
    os.path.exists(VAL_FEATURE_PATH)
):

    print("=" * 60)
    print("FEATURE FILES FOUND")
    print("=" * 60)

    extracted_features_train = pd.read_csv(
        TRAIN_FEATURE_PATH
    )

    extracted_features_val = pd.read_csv(
        VAL_FEATURE_PATH
    )

    print(
        "Training shape:",
        extracted_features_train.shape
    )

    print(
        "Validation shape:",
        extracted_features_val.shape
    )

else:

    print("=" * 60)
    print("FEATURE FILES NOT FOUND")
    print("EXTRACTING FEATURES...")
    print("=" * 60)

    MODEL_PATH = (
        "/content/drive/MyDrive/TinyBayes/run_2000/"
        "mobilenet_v3_small.tflite"
    )

    # --------------------------------------------------------
    # Load TFLite MobileNet
    # --------------------------------------------------------

    interpreter = tf.lite.Interpreter(
        model_path=MODEL_PATH
    )

    interpreter.allocate_tensors()

    input_details = (
        interpreter.get_input_details()
    )

    output_details = (
        interpreter.get_output_details()
    )

    # --------------------------------------------------------
    # Feature Extraction Function
    # --------------------------------------------------------

    def extract_full_image_features(
        dataframe,
        image_dir,
        training=True
    ):

        extracted_features = []

        for _, row in tqdm(
            dataframe.iterrows(),
            total=len(dataframe)
        ):

            image_path = os.path.join(
                image_dir,
                row["Image_ID"]
            )

            image = cv2.imread(
                image_path
            )

            if image is None:
                continue

            image = cv2.cvtColor(
                image,
                cv2.COLOR_BGR2RGB
            )

            image = cv2.resize(
                image,
                (224, 224)
            ).astype(np.float32)

            image /= 255.0

            image[:, :, 0] = (
                image[:, :, 0] - 0.485
            ) / 0.229

            image[:, :, 1] = (
                image[:, :, 1] - 0.456
            ) / 0.224

            image[:, :, 2] = (
                image[:, :, 2] - 0.406
            ) / 0.225

            input_tensor = np.expand_dims(
                image,
                axis=0
            ).astype(np.float32)

            interpreter.set_tensor(
                input_details[0]["index"],
                input_tensor
            )

            interpreter.invoke()

            feat = interpreter.get_tensor(
                output_details[0]["index"]
            )[0]

            if training:

                extracted_features.append([
                    *feat,
                    row["class"]
                ])

            else:

                extracted_features.append([
                    row["Image_ID"],
                    row["class"],
                    *feat
                ])

        feature_columns = [
            f"feat_{i}"
            for i in range(576)
        ]

        if training:

            feature_columns.append(
                "label"
            )

        else:

            feature_columns = (
                ["Image_ID", "class"]
                + feature_columns
            )

        return pd.DataFrame(
            extracted_features,
            columns=feature_columns
        )

    # --------------------------------------------------------
    # Training Features
    # --------------------------------------------------------

    print(
        "\nExtracting training features..."
    )

    extracted_features_train = (
        extract_full_image_features(
            df_train_balanced,
            IMAGE_DIR_TRAIN,
            training=True
        )
    )

    print(
        "Training shape:",
        extracted_features_train.shape
    )

    # --------------------------------------------------------
    # Validation Features
    # --------------------------------------------------------

    print(
        "\nExtracting validation features..."
    )

    extracted_features_val = (
        extract_full_image_features(
            validation_data,
            IMAGE_DIR_TRAIN,
            training=False
        )
    )

    print(
        "Validation shape:",
        extracted_features_val.shape
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    extracted_features_train.to_csv(
        TRAIN_FEATURE_PATH,
        index=False
    )

    extracted_features_val.to_csv(
        VAL_FEATURE_PATH,
        index=False
    )

    print("\nSaved:")

    print(TRAIN_FEATURE_PATH)

    print(VAL_FEATURE_PATH)

FEATURE FILES FOUND
Training shape: (2000, 577)
Validation shape: (500, 578)


In [ ]:
# ============================================================
# STEP 5
# TRAIN JACOBI-DMR
# ============================================================

from numpy.linalg import inv
import json

# ------------------------------------------------------------
# Training Data
# ------------------------------------------------------------

X_train = extracted_features_train.drop(
    columns=["label"]
)

Y_train = extracted_features_train["label"]

print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)

# ------------------------------------------------------------
# One-hot labels
# ------------------------------------------------------------

y_one_hot = pd.get_dummies(Y_train)

y_healthy = np.array(
    y_one_hot["healthy"]
)

y_cssvd = np.array(
    y_one_hot["cssvd"]
)

y_anthracnose = np.array(
    y_one_hot["anthracnose"]
)

# ------------------------------------------------------------
# Jacobi-DMR
# ------------------------------------------------------------

a = b = 1 / len(X_train)

k = 1

eta_healthy = np.log(
    (y_healthy + a) /
    (1 + k*b)
)

eta_cssvd = np.log(
    (y_cssvd + a) /
    (1 + k*b)
)

eta_anthracnose = np.log(
    (y_anthracnose + a) /
    (1 + k*b)
)

X = X_train.values

beta_healthy = (
    inv(X.T @ X)
    @
    (X.T @ eta_healthy)
)

beta_cssvd = (
    inv(X.T @ X)
    @
    (X.T @ eta_cssvd)
)

beta_anthracnose = (
    inv(X.T @ X)
    @
    (X.T @ eta_anthracnose)
)

# ------------------------------------------------------------
# Save coefficients
# ------------------------------------------------------------

jacobi_coeffs = {

    "healthy":
        beta_healthy.tolist(),

    "cssvd":
        beta_cssvd.tolist(),

    "anthracnose":
        beta_anthracnose.tolist()
}

JACOBI_PATH = os.path.join(
    SAVE_DIR,
    "jacobi_dmr_unique_images.json"
)

with open(
    JACOBI_PATH,
    "w"
) as f:

    json.dump(
        jacobi_coeffs,
        f
    )

print(
    "\nSaved:",
    JACOBI_PATH
)

X_train: (2000, 576)
Y_train: (2000,)

Saved: /content/drive/MyDrive/TinyBayes/run_2000/jacobi_dmr_unique_images.json


In [ ]:
# ============================================================
# STEP 6
# EVALUATE JACOBI-DMR
# ============================================================

import time

time_jacobi_start = time.time()

X_val = extracted_features_val.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

y_true = extracted_features_val[
    "class"
]

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

lambda_jacobi = pd.DataFrame({

    "healthy":
        np.exp(
            X_val @ beta_healthy
        ),

    "cssvd":
        np.exp(
            X_val @ beta_cssvd
        ),

    "anthracnose":
        np.exp(
            X_val @ beta_anthracnose
        )
})

y_pred_jacobi = lambda_jacobi.idxmax(
    axis=1
)

time_jacobi_dmr_seconds = (
    time.time() -
    time_jacobi_start
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

acc_jacobi = accuracy_score(
    y_true,
    y_pred_jacobi
)

cm_jacobi = confusion_matrix(
    y_true,
    y_pred_jacobi,
    labels=[
        "healthy",
        "cssvd",
        "anthracnose"
    ]
)

print("\n")
print("=" * 60)
print("JACOBI-DMR")
print("=" * 60)

print(
    f"Accuracy: {acc_jacobi:.4f}"
)

print(
    f"Time: {time_jacobi_dmr_seconds:.4f}s"
)

print(
    classification_report(
        y_true,
        y_pred_jacobi,
        labels=[
            "healthy",
            "cssvd",
            "anthracnose"
        ]
    )
)

print("\nConfusion Matrix:\n")
print(cm_jacobi)



JACOBI-DMR
Accuracy: 0.7840
Time: 0.0065s
              precision    recall  f1-score   support

     healthy       0.80      0.77      0.78       167
       cssvd       0.72      0.80      0.76       167
 anthracnose       0.84      0.79      0.81       166

    accuracy                           0.78       500
   macro avg       0.79      0.78      0.78       500
weighted avg       0.79      0.78      0.78       500


Confusion Matrix:

[[128  30   9]
 [ 18 133  16]
 [ 14  21 131]]


In [ ]:
# ============================================================
# RANDOM FOREST
# ============================================================

import time

time_rf_start = time.time()

X_train = extracted_features_train.drop(
    columns=["label"]
)

Y_train = extracted_features_train["label"]

X_val = extracted_features_val.drop(
    columns=["Image_ID", "class"]
)

y_true = extracted_features_val["class"]

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(
    X_train,
    Y_train
)

y_pred_rf = rf_model.predict(
    X_val
)

time_rf_seconds = (
    time.time() -
    time_rf_start
)

acc_rf = accuracy_score(
    y_true,
    y_pred_rf
)

cm_rf = confusion_matrix(
    y_true,
    y_pred_rf,
    labels=[
        "healthy",
        "cssvd",
        "anthracnose"
    ]
)

print("\n")
print("=" * 60)
print("RANDOM FOREST")
print("=" * 60)

print(
    f"Accuracy: {acc_rf:.4f}"
)

print(
    f"Time: {time_rf_seconds:.4f}s"
)

print(
    classification_report(
        y_true,
        y_pred_rf,
        labels=[
            "healthy",
            "cssvd",
            "anthracnose"
        ]
    )
)

print("\nConfusion Matrix:\n")
print(cm_rf)



RANDOM FOREST
Accuracy: 0.7260
Time: 6.4190s
              precision    recall  f1-score   support

     healthy       0.74      0.74      0.74       167
       cssvd       0.71      0.67      0.69       167
 anthracnose       0.73      0.77      0.75       166

    accuracy                           0.73       500
   macro avg       0.73      0.73      0.73       500
weighted avg       0.73      0.73      0.73       500


Confusion Matrix:

[[123  27  17]
 [ 25 112  30]
 [ 19  19 128]]


In [ ]:
# ============================================================
# SVM
# ============================================================

import time

time_svm_start = time.time()

X_train = extracted_features_train.drop(
    columns=["label"]
)

Y_train = extracted_features_train["label"]

X_val = extracted_features_val.drop(
    columns=["Image_ID", "class"]
)

y_true = extracted_features_val["class"]

svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale"
)

svm_model.fit(
    X_train,
    Y_train
)

y_pred_svm = svm_model.predict(
    X_val
)

time_svm_seconds = (
    time.time() -
    time_svm_start
)

acc_svm = accuracy_score(
    y_true,
    y_pred_svm
)

cm_svm = confusion_matrix(
    y_true,
    y_pred_svm,
    labels=[
        "healthy",
        "cssvd",
        "anthracnose"
    ]
)

print("\n")
print("=" * 60)
print("SVM")
print("=" * 60)

print(
    f"Accuracy: {acc_svm:.4f}"
)

print(
    f"Time: {time_svm_seconds:.4f}s"
)

print(
    classification_report(
        y_true,
        y_pred_svm,
        labels=[
            "healthy",
            "cssvd",
            "anthracnose"
        ]
    )
)

print("\nConfusion Matrix:\n")
print(cm_svm)



SVM
Accuracy: 0.8040
Time: 1.1423s
              precision    recall  f1-score   support

     healthy       0.81      0.83      0.82       167
       cssvd       0.77      0.78      0.78       167
 anthracnose       0.83      0.80      0.81       166

    accuracy                           0.80       500
   macro avg       0.80      0.80      0.80       500
weighted avg       0.80      0.80      0.80       500


Confusion Matrix:

[[139  20   8]
 [ 17 131  19]
 [ 15  19 132]]


In [ ]:
# ============================================================
# RIDGE CLASSIFIER
# ============================================================

import time

time_ridge_start = time.time()

X_train = extracted_features_train.drop(
    columns=["label"]
)

Y_train = extracted_features_train["label"]

X_val = extracted_features_val.drop(
    columns=["Image_ID", "class"]
)

y_true = extracted_features_val["class"]

ridge_model = RidgeClassifier(
    alpha=1.0
)

ridge_model.fit(
    X_train,
    Y_train
)

y_pred_ridge = ridge_model.predict(
    X_val
)

time_ridge_seconds = (
    time.time() -
    time_ridge_start
)

acc_ridge = accuracy_score(
    y_true,
    y_pred_ridge
)

cm_ridge = confusion_matrix(
    y_true,
    y_pred_ridge,
    labels=[
        "healthy",
        "cssvd",
        "anthracnose"
    ]
)

print("\n")
print("=" * 60)
print("RIDGE")
print("=" * 60)

print(
    f"Accuracy: {acc_ridge:.4f}"
)

print(
    f"Time: {time_ridge_seconds:.4f}s"
)

print(
    classification_report(
        y_true,
        y_pred_ridge,
        labels=[
            "healthy",
            "cssvd",
            "anthracnose"
        ]
    )
)

print("\nConfusion Matrix:\n")
print(cm_ridge)



RIDGE
Accuracy: 0.7940
Time: 0.1666s
              precision    recall  f1-score   support

     healthy       0.82      0.77      0.79       167
       cssvd       0.73      0.80      0.76       167
 anthracnose       0.84      0.81      0.83       166

    accuracy                           0.79       500
   macro avg       0.80      0.79      0.79       500
weighted avg       0.80      0.79      0.79       500


Confusion Matrix:

[[129  29   9]
 [ 18 133  16]
 [ 11  20 135]]


In [ ]:
# ============================================================
# LASSO (LOGISTIC L1)
# ============================================================

import time

time_lasso_start = time.time()

X_train = extracted_features_train.drop(
    columns=["label"]
)

Y_train = extracted_features_train["label"]

X_val = extracted_features_val.drop(
    columns=["Image_ID", "class"]
)

y_true = extracted_features_val["class"]

lasso_model = LogisticRegression(
    penalty="l1",
    solver="saga",
    max_iter=5000,
    random_state=42
)

lasso_model.fit(
    X_train,
    Y_train
)

y_pred_lasso = lasso_model.predict(
    X_val
)

time_lasso_seconds = (
    time.time() -
    time_lasso_start
)

acc_lasso = accuracy_score(
    y_true,
    y_pred_lasso
)

cm_lasso = confusion_matrix(
    y_true,
    y_pred_lasso,
    labels=[
        "healthy",
        "cssvd",
        "anthracnose"
    ]
)

print("\n")
print("=" * 60)
print("LASSO (LOGISTIC L1)")
print("=" * 60)

print(
    f"Accuracy: {acc_lasso:.4f}"
)

print(
    f"Time: {time_lasso_seconds:.4f}s"
)

print(
    classification_report(
        y_true,
        y_pred_lasso,
        labels=[
            "healthy",
            "cssvd",
            "anthracnose"
        ]
    )
)

print("\nConfusion Matrix:\n")
print(cm_lasso)



LASSO (LOGISTIC L1)
Accuracy: 0.7960
Time: 60.4123s
              precision    recall  f1-score   support

     healthy       0.82      0.78      0.80       167
       cssvd       0.76      0.77      0.77       167
 anthracnose       0.80      0.83      0.82       166

    accuracy                           0.80       500
   macro avg       0.80      0.80      0.80       500
weighted avg       0.80      0.80      0.80       500


Confusion Matrix:

[[131  25  11]
 [ 15 129  23]
 [ 13  15 138]]


In [ ]:
# ============================================================
# MODEL COMPARISON SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("FULL IMAGE + TFLITE MOBILENET FEATURE COMPARISON")
print("=" * 70)

print(
    f"{'Model':<30}"
    f"{'Accuracy':<15}"
    f"{'Time (s)':<15}"
)

print("-" * 70)

print(
    f"{'Jacobi-DMR':<30}"
    f"{acc_jacobi:<15.4f}"
    f"{time_jacobi_dmr_seconds:<15.4f}"
)

print(
    f"{'Random Forest':<30}"
    f"{acc_rf:<15.4f}"
    f"{time_rf_seconds:<15.4f}"
)

print(
    f"{'SVM (RBF)':<30}"
    f"{acc_svm:<15.4f}"
    f"{time_svm_seconds:<15.4f}"
)

print(
    f"{'Ridge Classifier':<30}"
    f"{acc_ridge:<15.4f}"
    f"{time_ridge_seconds:<15.4f}"
)

print(
    f"{'Lasso (Logistic L1)':<30}"
    f"{acc_lasso:<15.4f}"
    f"{time_lasso_seconds:<15.4f}"
)

print("=" * 70)



FULL IMAGE + TFLITE MOBILENET FEATURE COMPARISON
Model                         Accuracy       Time (s)       
----------------------------------------------------------------------
Jacobi-DMR                    0.7840         0.0089         
Random Forest                 0.7260         6.4190         
SVM (RBF)                     0.8040         1.1423         
Ridge Classifier              0.7940         0.1666         
Lasso (Logistic L1)           0.7960         60.4123        


In [ ]:
multi_class_images = (

    df_train.groupby("Image_ID")["class"]
    .nunique()
    .reset_index()

)

multi_class_images = multi_class_images[
    multi_class_images["class"] > 1
]

print(multi_class_images)

print("\nCount:", len(multi_class_images))

            Image_ID  class
154    ID_BRJQ51.JPG      2
854    ID_I9gkRe.JPG      2
4207   ID_nRsOlR.JPG      2
4577   ID_r1LWke.JPG      2
4911  ID_uKHtOl.jpeg      2
5390  ID_yeLQCq.jpeg      2

Count: 6


In [ ]:
problem_images = multi_class_images["Image_ID"]

df_train[
    df_train["Image_ID"].isin(problem_images)
].sort_values("Image_ID")

,Image_ID,class,confidence,ymin,xmin,ymax,xmax,class_id,ImagePath
5712,ID_BRJQ51.JPG,cssvd,1.0,129.0,260.0,2877.0,1122.0,1,dataset/images/train/ID_BRJQ51.JPG
4010,ID_BRJQ51.JPG,healthy,1.0,26.0,73.0,2809.0,1647.0,2,dataset/images/train/ID_BRJQ51.JPG
3632,ID_I9gkRe.JPG,healthy,1.0,20.0,54.0,3264.0,2076.0,2,dataset/images/train/ID_I9gkRe.JPG
5616,ID_I9gkRe.JPG,cssvd,1.0,75.0,260.0,3264.0,1970.0,1,dataset/images/train/ID_I9gkRe.JPG
3633,ID_I9gkRe.JPG,healthy,1.0,0.0,1731.0,1834.0,2448.0,2,dataset/images/train/ID_I9gkRe.JPG
3992,ID_nRsOlR.JPG,healthy,1.0,20.0,21.0,2620.0,1146.0,2,dataset/images/train/ID_nRsOlR.JPG
5629,ID_nRsOlR.JPG,cssvd,1.0,0.0,993.0,2633.0,1991.0,1,dataset/images/train/ID_nRsOlR.JPG
3993,ID_nRsOlR.JPG,healthy,1.0,0.0,1022.0,2919.0,2448.0,2,dataset/images/train/ID_nRsOlR.JPG
4328,ID_r1LWke.JPG,healthy,1.0,13.0,0.0,3264.0,1939.0,2,dataset/images/train/ID_r1LWke.JPG
5723,ID_r1LWke.JPG,cssvd,1.0,75.0,0.0,3264.0,1699.0,1,dataset/images/train/ID_r1LWke.JPG


# **Fine-tune MobileNet**

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import cv2
import torch
import numpy as np
import pandas as pd

from PIL import Image
from tqdm import tqdm

import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from torchvision import transforms

from torchvision.models import (
    mobilenet_v3_small,
    MobileNet_V3_Small_Weights
)

from sklearn.metrics import (
    accuracy_score,
    classification_report
)

Mounted at /content/drive


In [ ]:
# ============================================================
# PATHS
# ============================================================

SAVE_DIR = (
    "/content/drive/MyDrive/TinyBayes/run_2000"
)

DATASET_ROOT = (
    "/content/drive/MyDrive/TinyBayes/amini_dataset"
)

BASE_DIR = os.path.join(
    DATASET_ROOT,
    "dataset"
)

IMAGE_DIR_TRAIN = os.path.join(
    BASE_DIR,
    "images",
    "train"
)

IMAGE_DIR_TEST = os.path.join(
    BASE_DIR,
    "images",
    "test"
)

IMAGE_DIR_VAL = os.path.join(
    BASE_DIR,
    "images",
    "val"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

In [ ]:
# ============================================================
# CREATE OR LOAD FINETUNE SET
# ============================================================

TRAIN_CSV = os.path.join(
    SAVE_DIR,
    "train_2000_unique_images.csv"
)

VAL_CSV = os.path.join(
    SAVE_DIR,
    "val_500_unique_images.csv"
)

FINETUNE_CSV = os.path.join(
    SAVE_DIR,
    "finetune_500_unique_images.csv"
)

train_df = pd.read_csv(
    TRAIN_CSV
)

val_df = pd.read_csv(
    VAL_CSV
)

# ------------------------------------------------------------
# Load if already exists
# ------------------------------------------------------------

if os.path.exists(FINETUNE_CSV):

    print(
        "Finetune split already exists."
    )

    finetune_data = pd.read_csv(
        FINETUNE_CSV
    )

    print(
        "Loaded:",
        FINETUNE_CSV
    )

    print("\nClass Distribution:")

    print(
        finetune_data["class"]
        .value_counts()
    )

# ------------------------------------------------------------
# Otherwise create it
# ------------------------------------------------------------

else:

    print(
        "Creating finetune split..."
    )

    # --------------------------------------------------------
    # Build clean image-level dataset
    # --------------------------------------------------------

    image_level_df = (
        df_train[
            ["Image_ID", "class"]
        ]
        .drop_duplicates()
    )

    # --------------------------------------------------------
    # Remove ambiguous images
    # --------------------------------------------------------

    image_class_count = (
        image_level_df
        .groupby("Image_ID")["class"]
        .nunique()
    )

    ambiguous_images = set(
        image_class_count[
            image_class_count > 1
        ].index
    )

    clean_df = image_level_df[
        ~image_level_df["Image_ID"].isin(
            ambiguous_images
        )
    ].reset_index(
        drop=True
    )

    print(
        "Clean images:",
        len(clean_df)
    )

    # --------------------------------------------------------
    # Remove train + validation images
    # --------------------------------------------------------

    used_images = set(
        train_df["Image_ID"]
    ).union(
        set(
            val_df["Image_ID"]
        )
    )

    remaining_images = clean_df[
        ~clean_df["Image_ID"].isin(
            used_images
        )
    ].reset_index(
        drop=True
    )

    print(
        "Remaining images:",
        len(remaining_images)
    )

    # --------------------------------------------------------
    # Create balanced finetune set
    # --------------------------------------------------------

    FINETUNE_SIZE = 500

    per_class = (
        FINETUNE_SIZE // 3
    )

    parts = []

    for cls in [
        "healthy",
        "cssvd",
        "anthracnose"
    ]:

        cls_df = remaining_images[
            remaining_images["class"] == cls
        ]

        sampled = cls_df.sample(
            n=per_class,
            random_state=123
        )

        parts.append(
            sampled
        )

    finetune_data = pd.concat(
        parts
    ).reset_index(
        drop=True
    )

    remaining_needed = (
        FINETUNE_SIZE -
        len(finetune_data)
    )

    if remaining_needed > 0:

        remaining_pool = remaining_images[
            ~remaining_images["Image_ID"].isin(
                finetune_data["Image_ID"]
            )
        ]

        extra = remaining_pool.sample(
            n=remaining_needed,
            random_state=123
        )

        finetune_data = pd.concat([
            finetune_data,
            extra
        ]).reset_index(
            drop=True
        )

    # --------------------------------------------------------
    # Verify no overlap
    # --------------------------------------------------------

    overlap_train = set(
        train_df["Image_ID"]
    ).intersection(
        set(
            finetune_data["Image_ID"]
        )
    )

    overlap_val = set(
        val_df["Image_ID"]
    ).intersection(
        set(
            finetune_data["Image_ID"]
        )
    )

    print(
        "\nTrain-Finetune overlap:",
        len(overlap_train)
    )

    print(
        "Val-Finetune overlap:",
        len(overlap_val)
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    finetune_data.to_csv(
        FINETUNE_CSV,
        index=False
    )

    print("\nClass Distribution:")

    print(
        finetune_data["class"]
        .value_counts()
    )

    print(
        "\nSaved:",
        FINETUNE_CSV
    )

Finetune split already exists.
Loaded: /content/drive/MyDrive/TinyBayes/run_2000/finetune_500_unique_images.csv

Class Distribution:
class
healthy        167
cssvd          167
anthracnose    166
Name: count, dtype: int64


In [ ]:
# ============================================================
# LOAD SPLITS
# ============================================================

TRAIN_CSV = os.path.join(
    SAVE_DIR,
    "train_2000_unique_images.csv"
)

VAL_CSV = os.path.join(
    SAVE_DIR,
    "val_500_unique_images.csv"
)

FINETUNE_CSV = os.path.join(
    SAVE_DIR,
    "finetune_500_unique_images.csv"
)

df_train_balanced = pd.read_csv(
    TRAIN_CSV
)

validation_data = pd.read_csv(
    VAL_CSV
)

finetune_data = pd.read_csv(
    FINETUNE_CSV
)

print(
    "Train:",
    df_train_balanced.shape
)

print(
    "Validation:",
    validation_data.shape
)

print(
    "FineTune:",
    finetune_data.shape
)

Train: (2000, 2)
Validation: (500, 2)
FineTune: (500, 2)


In [ ]:
# ============================================================
# LABEL MAP
# ============================================================

label_map = {
    "healthy": 0,
    "cssvd": 1,
    "anthracnose": 2
}

reverse_label_map = {
    0: "healthy",
    1: "cssvd",
    2: "anthracnose"
}

In [ ]:
# ============================================================
# DATASET
# ============================================================

class CocoaDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_dir,
        transform=None
    ):

        self.df = dataframe
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_path = os.path.join(
            self.image_dir,
            row["Image_ID"]
        )

        image = Image.open(
            image_path
        ).convert("RGB")

        if self.transform:

            image = self.transform(
                image
            )

        label = label_map[
            row["class"]
        ]

        return image, label

In [ ]:
# ============================================================
# TRANSFORM
# ============================================================

transform = transforms.Compose([

    transforms.Resize(
        (224,224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

In [ ]:
# ============================================================
# DATALOADERS
# ============================================================

train_dataset = CocoaDataset(
    df_train_balanced,
    IMAGE_DIR_TRAIN,
    transform
)

finetune_dataset = CocoaDataset(
    finetune_data,
    IMAGE_DIR_TRAIN,
    transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

finetune_loader = DataLoader(
    finetune_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

In [ ]:
# ============================================================
# LOAD MODEL
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = mobilenet_v3_small(
    weights=
    MobileNet_V3_Small_Weights.DEFAULT
)

num_features = (
    model.classifier[3]
    .in_features
)

model.classifier[3] = nn.Linear(
    num_features,
    3
)

model = model.to(device)

print(model)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 104MB/s]

MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), 

In [ ]:
# ============================================================
# TRAINING SETUP
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [ ]:
# ============================================================
# FINE-TUNE MOBILENET
# SKIP IF ALREADY TRAINED
# ============================================================

EPOCHS = 5

BEST_MODEL_PATH = os.path.join(
    SAVE_DIR,
    "mobilenet_finetuned.pth"
)

# ------------------------------------------------------------
# Check if model already exists
# ------------------------------------------------------------

if os.path.exists(BEST_MODEL_PATH):

    print(
        "Fine-tuned model already exists."
    )

    print(
        "Loading:",
        BEST_MODEL_PATH
    )

    model.load_state_dict(
        torch.load(
            BEST_MODEL_PATH,
            map_location=device
        )
    )

    model = model.to(device)

else:

    print(
        "No saved model found."
    )

    print(
        "Starting fine-tuning..."
    )

    best_acc = 0

    for epoch in range(EPOCHS):

        # ----------------------------------
        # Training
        # ----------------------------------

        model.train()

        train_loss = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            train_loss += loss.item()

        # ----------------------------------
        # Validation
        # ----------------------------------

        model.eval()

        y_true = []
        y_pred = []

        with torch.no_grad():

            for images, labels in finetune_loader:

                images = images.to(device)

                outputs = model(images)

                preds = torch.argmax(
                    outputs,
                    dim=1
                )

                y_true.extend(
                    labels.numpy()
                )

                y_pred.extend(
                    preds.cpu().numpy()
                )

        acc = accuracy_score(
            y_true,
            y_pred
        )

        print(
            f"Epoch {epoch+1}"
            f" | Acc = {acc:.4f}"
        )

        if acc > best_acc:

            best_acc = acc

            torch.save(
                model.state_dict(),
                BEST_MODEL_PATH
            )

        print(
            f"Best so far: {best_acc:.4f}"
        )

    print("\nTraining Complete")

    print(
        "Best Accuracy:",
        best_acc
    )

    print(
        "Saved:",
        BEST_MODEL_PATH
    )

    # Reload best model

    model.load_state_dict(
        torch.load(
            BEST_MODEL_PATH,
            map_location=device
        )
    )

    model = model.to(device)

print(
    "\nModel ready."
)

Fine-tuned model already exists.
Loading: /content/drive/MyDrive/TinyBayes/run_2000/mobilenet_finetuned.pth

Model ready.


In [ ]:
# ============================================================
# LOAD FINE-TUNED FEATURE EXTRACTOR
# ============================================================

import torch
import torch.nn as nn

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

BEST_MODEL_PATH = os.path.join(
    SAVE_DIR,
    "mobilenet_finetuned.pth"
)

model = mobilenet_v3_small(
    weights=None
)

num_features = (
    model.classifier[3]
    .in_features
)

model.classifier[3] = nn.Linear(
    num_features,
    3
)

model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=device
    )
)

model = model.to(device)

print(
    "Fine-tuned model loaded."
)

Fine-tuned model loaded.


In [ ]:
# ============================================================
# CONVERT TO FEATURE EXTRACTOR
# ============================================================

feature_extractor = nn.Sequential(

    model.features,

    model.avgpool,

    nn.Flatten()

)

feature_extractor.eval()

print(
    feature_extractor
)

Sequential(
  (0): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(

In [ ]:
dummy = torch.randn(
    1,
    3,
    224,
    224
).to(device)

with torch.no_grad():

    feat = feature_extractor(
        dummy
    )

print(feat.shape)

torch.Size([1, 576])


In [ ]:
# ============================================================
# FEATURE EXTRACTION FUNCTION
# ============================================================

from PIL import Image

transform = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


def extract_finetuned_features(
    dataframe,
    image_dir,
    training=True
):

    extracted_features = []

    with torch.no_grad():

        for _, row in tqdm(
            dataframe.iterrows(),
            total=len(dataframe)
        ):

            image_path = os.path.join(
                image_dir,
                row["Image_ID"]
            )

            try:

                image = Image.open(
                    image_path
                ).convert("RGB")

            except:

                continue

            image = transform(
                image
            )

            image = image.unsqueeze(
                0
            ).to(device)

            features = (
                feature_extractor(
                    image
                )
                .cpu()
                .numpy()[0]
            )

            if training:

                extracted_features.append([

                    *features,

                    row["class"]

                ])

            else:

                extracted_features.append([

                    row["Image_ID"],

                    row["class"],

                    *features

                ])

    feature_columns = [

        f"feat_{i}"

        for i in range(576)

    ]

    if training:

        feature_columns.append(
            "label"
        )

    else:

        feature_columns = (
            ["Image_ID", "class"]
            + feature_columns
        )

    return pd.DataFrame(

        extracted_features,

        columns=feature_columns

    )

In [ ]:
# ============================================================
# LOAD OR CREATE FEATURES
# ============================================================

FINETUNED_TRAIN_PATH = os.path.join(
    SAVE_DIR,
    "finetuned_train_features.csv"
)

FINETUNED_VAL_PATH = os.path.join(
    SAVE_DIR,
    "finetuned_val_features.csv"
)

if (
    os.path.exists(FINETUNED_TRAIN_PATH)
    and
    os.path.exists(FINETUNED_VAL_PATH)
):

    print(
        "Feature files already exist."
    )

    finetuned_train_features = pd.read_csv(
        FINETUNED_TRAIN_PATH
    )

    finetuned_val_features = pd.read_csv(
        FINETUNED_VAL_PATH
    )

else:

    print(
        "Extracting training features..."
    )

    finetuned_train_features = (
        extract_finetuned_features(
            df_train_balanced,
            IMAGE_DIR_TRAIN,
            training=True
        )
    )

    print(
        finetuned_train_features.shape
    )

    print(
        "\nExtracting validation features..."
    )

    finetuned_val_features = (
        extract_finetuned_features(
            validation_data,
            IMAGE_DIR_TRAIN,
            training=False
        )
    )

    print(
        finetuned_val_features.shape
    )

    finetuned_train_features.to_csv(
        FINETUNED_TRAIN_PATH,
        index=False
    )

    finetuned_val_features.to_csv(
        FINETUNED_VAL_PATH,
        index=False
    )

    print(
        "\nSaved:"
    )

    print(
        FINETUNED_TRAIN_PATH
    )

    print(
        FINETUNED_VAL_PATH
    )

Feature files already exist.


In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

time_jacobi_dmr_ft = time.time()

X_train = finetuned_train_features.drop(
    columns=["label"]
)

Y_train = finetuned_train_features["label"]

X_val = finetuned_val_features.drop(
    columns=["Image_ID", "class"]
)

y_true = finetuned_val_features["class"]

# ------------------------------------------------------------
# One-hot labels
# ------------------------------------------------------------

y_one_hot = pd.get_dummies(
    Y_train
)

y_healthy = np.array(
    y_one_hot["healthy"]
)

y_cssvd = np.array(
    y_one_hot["cssvd"]
)

y_anthracnose = np.array(
    y_one_hot["anthracnose"]
)

# ------------------------------------------------------------
# Jacobi prior
# ------------------------------------------------------------

a = b = 1 / len(X_train)
k = 1

eta_healthy = np.log(
    (y_healthy + a)
    /
    (1 + k * b)
)

eta_cssvd = np.log(
    (y_cssvd + a)
    /
    (1 + k * b)
)

eta_anthracnose = np.log(
    (y_anthracnose + a)
    /
    (1 + k * b)
)

# ------------------------------------------------------------
# Stable coefficient estimation
# ------------------------------------------------------------

X = X_train.values.astype(
    np.float64
)

XTX_inv = np.linalg.pinv(
    X.T @ X
)

beta_healthy = (
    XTX_inv
    @
    (X.T @ eta_healthy)
)

beta_cssvd = (
    XTX_inv
    @
    (X.T @ eta_cssvd)
)

beta_anthracnose = (
    XTX_inv
    @
    (X.T @ eta_anthracnose)
)

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

lambda_jacobi = pd.DataFrame({

    "healthy":
        np.exp(
            X_val @ beta_healthy
        ),

    "cssvd":
        np.exp(
            X_val @ beta_cssvd
        ),

    "anthracnose":
        np.exp(
            X_val @ beta_anthracnose
        )
})

y_pred = lambda_jacobi.idxmax(
    axis=1
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

acc_jacobi_ft = accuracy_score(
    y_true,
    y_pred
)

time_jacobi_dmr_ft = (
    time.time()
    - time_jacobi_dmr_ft
)

print("\n")
print("=" * 60)
print("JACOBI-DMR (FINETUNED FEATURES)")
print("=" * 60)

print(
    f"Accuracy: {acc_jacobi_ft:.4f}"
)

print(
    f"Time: {time_jacobi_dmr_ft:.4f}s"
)

print(
    classification_report(
        y_true,
        y_pred
    )
)

print(
    confusion_matrix(
        y_true,
        y_pred
    )
)



JACOBI-DMR (FINETUNED FEATURES)
Accuracy: 0.8420
Time: 0.3445s
              precision    recall  f1-score   support

 anthracnose       0.88      0.86      0.87       166
       cssvd       0.79      0.81      0.80       167
     healthy       0.85      0.86      0.86       167

    accuracy                           0.84       500
   macro avg       0.84      0.84      0.84       500
weighted avg       0.84      0.84      0.84       500

[[142  17   7]
 [ 14 135  18]
 [  5  18 144]]


In [ ]:
import json

jacobi_coeffs = {
    "healthy": beta_healthy.tolist(),
    "cssvd": beta_cssvd.tolist(),
    "anthracnose": beta_anthracnose.tolist()
}

NEW_JACOBI_PATH = os.path.join(
    SAVE_DIR,
    "jacobi_dmr_finetuned.json"
)

with open(NEW_JACOBI_PATH, "w") as f:
    json.dump(jacobi_coeffs, f)

print("Saved:", NEW_JACOBI_PATH)

Saved: /content/drive/MyDrive/TinyBayes/run_2000/jacobi_dmr_finetuned.json


In [ ]:
import json

with open(
    "/content/drive/MyDrive/TinyBayes/run_2000/jacobi_dmr_finetuned.json"
) as f:

    coeffs = json.load(f)

print(len(coeffs["healthy"]))
print(len(coeffs["cssvd"]))
print(len(coeffs["anthracnose"]))

576
576
576


In [ ]:
# ============================================================
# RANDOM FOREST (FINETUNED FEATURES)
# ============================================================

import time

time_rf_ft_start = time.time()

X_train = finetuned_train_features.drop(
    columns=["label"]
)

Y_train = finetuned_train_features["label"]

X_val = finetuned_val_features.drop(
    columns=["Image_ID", "class"]
)

y_true = finetuned_val_features["class"]

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(
    X_train,
    Y_train
)

y_pred_rf_ft = rf_model.predict(
    X_val
)

time_rf_ft = (
    time.time()
    - time_rf_ft_start
)

acc_rf_ft = accuracy_score(
    y_true,
    y_pred_rf_ft
)

print("\n")
print("=" * 60)
print("RANDOM FOREST (FINETUNED FEATURES)")
print("=" * 60)

print(
    f"Accuracy: {acc_rf_ft:.4f}"
)

print(
    f"Time: {time_rf_ft:.4f}s"
)

print(
    classification_report(
        y_true,
        y_pred_rf_ft
    )
)

print(
    confusion_matrix(
        y_true,
        y_pred_rf_ft
    )
)



RANDOM FOREST (FINETUNED FEATURES)
Accuracy: 0.8640
Time: 5.5609s
              precision    recall  f1-score   support

 anthracnose       0.92      0.87      0.90       166
       cssvd       0.82      0.84      0.83       167
     healthy       0.85      0.88      0.87       167

    accuracy                           0.86       500
   macro avg       0.87      0.86      0.86       500
weighted avg       0.87      0.86      0.86       500

[[145  12   9]
 [ 11 140  16]
 [  2  18 147]]


In [ ]:
# ============================================================
# SVM (FINETUNED FEATURES)
# ============================================================

import time

time_svm_ft_start = time.time()

X_train = finetuned_train_features.drop(
    columns=["label"]
)

Y_train = finetuned_train_features["label"]

X_val = finetuned_val_features.drop(
    columns=["Image_ID", "class"]
)

y_true = finetuned_val_features["class"]

svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale"
)

svm_model.fit(
    X_train,
    Y_train
)

y_pred_svm_ft = svm_model.predict(
    X_val
)

time_svm_ft = (
    time.time()
    - time_svm_ft_start
)

acc_svm_ft = accuracy_score(
    y_true,
    y_pred_svm_ft
)

print("\n")
print("=" * 60)
print("SVM (FINETUNED FEATURES)")
print("=" * 60)

print(
    f"Accuracy: {acc_svm_ft:.4f}"
)

print(
    f"Time: {time_svm_ft:.4f}s"
)

print(
    classification_report(
        y_true,
        y_pred_svm_ft
    )
)

print(
    confusion_matrix(
        y_true,
        y_pred_svm_ft
    )
)



SVM (FINETUNED FEATURES)
Accuracy: 0.8620
Time: 0.4053s
              precision    recall  f1-score   support

 anthracnose       0.92      0.86      0.89       166
       cssvd       0.80      0.86      0.83       167
     healthy       0.87      0.87      0.87       167

    accuracy                           0.86       500
   macro avg       0.86      0.86      0.86       500
weighted avg       0.86      0.86      0.86       500

[[143  14   9]
 [ 11 143  13]
 [  1  21 145]]


In [ ]:
# ============================================================
# RIDGE (FINETUNED FEATURES)
# ============================================================

import time

time_ridge_ft_start = time.time()

X_train = finetuned_train_features.drop(
    columns=["label"]
)

Y_train = finetuned_train_features["label"]

X_val = finetuned_val_features.drop(
    columns=["Image_ID", "class"]
)

y_true = finetuned_val_features["class"]

ridge_model = RidgeClassifier(
    alpha=1.0
)

ridge_model.fit(
    X_train,
    Y_train
)

y_pred_ridge_ft = ridge_model.predict(
    X_val
)

time_ridge_ft = (
    time.time()
    - time_ridge_ft_start
)

acc_ridge_ft = accuracy_score(
    y_true,
    y_pred_ridge_ft
)

print("\n")
print("=" * 60)
print("RIDGE (FINETUNED FEATURES)")
print("=" * 60)

print(
    f"Accuracy: {acc_ridge_ft:.4f}"
)

print(
    f"Time: {time_ridge_ft:.4f}s"
)

print(
    classification_report(
        y_true,
        y_pred_ridge_ft
    )
)

print(
    confusion_matrix(
        y_true,
        y_pred_ridge_ft
    )
)



RIDGE (FINETUNED FEATURES)
Accuracy: 0.8460
Time: 0.3963s
              precision    recall  f1-score   support

 anthracnose       0.91      0.84      0.87       166
       cssvd       0.80      0.83      0.81       167
     healthy       0.84      0.87      0.86       167

    accuracy                           0.85       500
   macro avg       0.85      0.85      0.85       500
weighted avg       0.85      0.85      0.85       500

[[139  17  10]
 [ 11 138  18]
 [  3  18 146]]


In [ ]:
# ============================================================
# LASSO (FINETUNED FEATURES)
# ============================================================

import time

time_lasso_ft_start = time.time()

X_train = finetuned_train_features.drop(
    columns=["label"]
)

Y_train = finetuned_train_features["label"]

X_val = finetuned_val_features.drop(
    columns=["Image_ID", "class"]
)

y_true = finetuned_val_features["class"]

lasso_model = LogisticRegression(
    penalty="l1",
    solver="saga",
    max_iter=5000,
    random_state=42
)

lasso_model.fit(
    X_train,
    Y_train
)

y_pred_lasso_ft = lasso_model.predict(
    X_val
)

time_lasso_ft = (
    time.time()
    - time_lasso_ft_start
)

acc_lasso_ft = accuracy_score(
    y_true,
    y_pred_lasso_ft
)

print("\n")
print("=" * 60)
print("LASSO (FINETUNED FEATURES)")
print("=" * 60)

print(
    f"Accuracy: {acc_lasso_ft:.4f}"
)

print(
    f"Time: {time_lasso_ft:.4f}s"
)

print(
    classification_report(
        y_true,
        y_pred_lasso_ft
    )
)

print(
    confusion_matrix(
        y_true,
        y_pred_lasso_ft
    )
)



LASSO (FINETUNED FEATURES)
Accuracy: 0.8460
Time: 94.6548s
              precision    recall  f1-score   support

 anthracnose       0.89      0.87      0.88       166
       cssvd       0.79      0.80      0.79       167
     healthy       0.86      0.87      0.87       167

    accuracy                           0.85       500
   macro avg       0.85      0.85      0.85       500
weighted avg       0.85      0.85      0.85       500

[[144  16   6]
 [ 16 133  18]
 [  2  19 146]]


In [ ]:
# ============================================================
# FINETUNED MOBILENET FEATURE COMPARISON
# ============================================================

print("\n")
print("=" * 70)
print("FINETUNED MOBILENET FEATURE COMPARISON")
print("=" * 70)

print(
    f"{'Model':<30}"
    f"{'Accuracy':<15}"
    f"{'Time (s)':<15}"
)

print("-" * 70)

print(
    f"{'Jacobi-DMR':<30}"
    f"{acc_jacobi_ft:<15.4f}"
    f"{time_jacobi_dmr_ft:<15.4f}"
)

print(
    f"{'Random Forest':<30}"
    f"{acc_rf_ft:<15.4f}"
    f"{time_rf_ft:<15.4f}"
)

print(
    f"{'SVM (RBF)':<30}"
    f"{acc_svm_ft:<15.4f}"
    f"{time_svm_ft:<15.4f}"
)

print(
    f"{'Ridge Classifier':<30}"
    f"{acc_ridge_ft:<15.4f}"
    f"{time_ridge_ft:<15.4f}"
)

print(
    f"{'Lasso (Logistic L1)':<30}"
    f"{acc_lasso_ft:<15.4f}"
    f"{time_lasso_ft:<15.4f}"
)

print("=" * 70)



FINETUNED MOBILENET FEATURE COMPARISON
Model                         Accuracy       Time (s)       
----------------------------------------------------------------------
Jacobi-DMR                    0.8420         0.9265         
Random Forest                 0.8640         5.5609         
SVM (RBF)                     0.8620         0.4053         
Ridge Classifier              0.8460         0.3963         
Lasso (Logistic L1)           0.8460         94.6548        


Converting Finetuned MobileNet to tflite

In [ ]:
print(feat.shape)

torch.Size([1, 576])


In [ ]:

!pip install -q onnx onnxruntime onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 7.7 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = mobilenet_v3_small(
    weights=None
)

num_features = (
    model.classifier[3]
    .in_features
)

model.classifier[3] = nn.Linear(
    num_features,
    3
)

model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=device
    )
)

model.eval()

feature_extractor = nn.Sequential(
    model.features,
    model.avgpool,
    nn.Flatten()
)

feature_extractor.eval()

Sequential(
  (0): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(

In [ ]:
import torch

ONNX_PATH = os.path.join(
    SAVE_DIR,
    "mobilenet_finetuned_576.onnx"
)

dummy_input = torch.randn(
    1,
    3,
    224,
    224
)

torch.onnx.export(
    feature_extractor,
    dummy_input,
    ONNX_PATH,
    input_names=["input"],
    output_names=["features"],
    opset_version=18
)

print("Saved:", ONNX_PATH)

[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Saved: /content/drive/MyDrive/TinyBayes/run_2000/mobilenet_finetuned_576.onnx


In [ ]:
import onnx
import onnxruntime as ort
import numpy as np

ONNX_PATH = os.path.join(
    SAVE_DIR,
    "mobilenet_finetuned_576.onnx"
)

model = onnx.load(ONNX_PATH)

onnx.checker.check_model(model)

print("ONNX model is valid")

session = ort.InferenceSession(
    ONNX_PATH,
    providers=["CPUExecutionProvider"]
)

dummy = np.random.randn(
    1,
    3,
    224,
    224
).astype(np.float32)

output = session.run(
    None,
    {"input": dummy}
)

print("Output shape:", output[0].shape)

ONNX model is valid
Output shape: (1, 576)


In [ ]:
import numpy as np
import onnxruntime as ort

# random test image
x = torch.randn(
    1,
    3,
    224,
    224
)

with torch.no_grad():

    pt_features = (
        feature_extractor(x)
        .cpu()
        .numpy()
    )

session = ort.InferenceSession(
    ONNX_PATH,
    providers=["CPUExecutionProvider"]
)

onnx_features = session.run(
    None,
    {"input": x.numpy()}
)[0]

print(
    "PyTorch shape:",
    pt_features.shape
)

print(
    "ONNX shape:",
    onnx_features.shape
)

print(
    "Max Difference:",
    np.max(
        np.abs(
            pt_features -
            onnx_features
        )
    )
)

cosine = np.dot(
    pt_features.flatten(),
    onnx_features.flatten()
) / (
    np.linalg.norm(
        pt_features.flatten()
    )
    *
    np.linalg.norm(
        onnx_features.flatten()
    )
)

print(
    "Cosine Similarity:",
    cosine
)

PyTorch shape: (1, 576)
ONNX shape: (1, 576)
Max Difference: 3.2186508e-06
Cosine Similarity: 1.0000001


In [ ]:
import importlib

packages = [
    "onnx",
    "onnxruntime",
    "onnxscript",
    "onnx2tf",
    "onnx_graphsurgeon",
    "tensorflow",
    "ai_edge_litert"
]

for p in packages:

    try:
        importlib.import_module(p)

        print(
            f"{p}: INSTALLED"
        )

    except:

        print(
            f"{p}: NOT INSTALLED"
        )

onnx: INSTALLED
onnxruntime: INSTALLED
onnxscript: INSTALLED
onnx2tf: NOT INSTALLED
onnx_graphsurgeon: NOT INSTALLED
tensorflow: INSTALLED
ai_edge_litert: NOT INSTALLED


In [ ]:
ONNX_PATH = os.path.join(
    SAVE_DIR,
    "mobilenet_finetuned_576.onnx"
)

torch.onnx.export(
    feature_extractor,
    dummy_input,
    ONNX_PATH,
    input_names=["input"],
    output_names=["features"],
    opset_version=18
)

print("Saved:", ONNX_PATH)

RuntimeError: Decorator script does not work on dynamically compiled function aten_zeros.

In [ ]:
!pip install -q onnxsim
!pip install -q ai-edge-litert
!pip install -q onnx_graphsurgeon
!pip install -q sng4onnx
!pip install -q onnx2tf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31

In [ ]:
from onnx2tf import convert

print("onnx2tf imported successfully")

import numpy as np
import tensorflow as tf

print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)

onnx2tf imported successfully
NumPy: 2.0.2
TensorFlow: 2.20.0


In [ ]:
from onnx2tf import convert

convert(
    input_onnx_file_path=
    "/content/drive/MyDrive/TinyBayes/run_2000/mobilenet_finetuned_576.onnx",

    output_folder_path=
    "/content/drive/MyDrive/TinyBayes/run_2000/mobilenet_finetuned_saved_model"
)


Model optimizing started ============================================================
Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃             ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Add         │ 6              │ 6                │
│ Constant    │ 106            │ 106              │
│ Conv        │ 52             │ 52               │
│ HardSigmoid │ 9              │ 9                │
│ HardSwish   │ 18             │ 18               │
│ Mul         │ 9              │ 9                │
│ ReduceMean  │ 10             │ 10               │
│ Relu        │ 14             │ 14               │
│ Reshape     │ 1              │ 1                │
│ Model Size  │ 3.8MiB         │ 3.5MiB           │
└─────────────┴────────────────┴──────────────────┘

Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃             ┃ Original Model ┃ Simpl

flatbuffer_direct lowering:   0%|          | 0/119 [00:00<?, ?it/s]

flatbuffer_direct post-lowering:   0%|          | 0/6 [00:00<?, ?it/s]

flatbuffer_direct export:   0%|          | 0/3 [00:00<?, ?it/s]

flatbuffer_direct write timing: stage=float32 mode=builder_direct total=0.049s serialize=0.027s (sanitize=0.001s build=0.004s pack=0.022s output=0.001s) write=0.019s size=3.55MB
flatbuffer_direct write timing: stage=float16 mode=builder_direct total=0.039s serialize=0.023s (sanitize=0.000s build=0.003s pack=0.020s output=0.000s) write=0.014s size=1.80MB
Float32 tflite output complete! (/content/drive/MyDrive/TinyBayes/run_2000/mobilenet_finetuned_saved_model/mobilenet_finetuned_576_float32.tflite)
Float16 tflite output complete! (/content/drive/MyDrive/TinyBayes/run_2000/mobilenet_finetuned_saved_model/mobilenet_finetuned_576_float16.tflite)
Tensor correspondence report output complete! (/content/drive/MyDrive/TinyBayes/run_2000/mobilenet_finetuned_saved_model/mobilenet_finetuned_576_tensor_correspondence_report.json)


In [ ]:
import torch
import tensorflow as tf
import sys

print("Torch:", torch.__version__)
print("CUDA :", torch.version.cuda)
print("Python:", sys.version)

print("TensorFlow:", tf.__version__)

Torch: 2.11.0+cpu
CUDA : None
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
TensorFlow: 2.20.0


In [ ]:
import torch
import os

EXPORT_PATH = os.path.join(
    SAVE_DIR,
    "mobilenet_finetuned_576.pt"
)

feature_extractor.eval()

example = torch.randn(
    1,
    3,
    224,
    224
)

traced_model = torch.jit.trace(
    feature_extractor,
    example
)

traced_model.save(
    EXPORT_PATH
)

print("Saved:", EXPORT_PATH)

In [ ]:
import torch
from torch.utils.mobile_optimizer import optimize_for_mobile

model = torch.jit.load(
    "/content/drive/MyDrive/TinyBayes/run_2000/mobilenet_finetuned_576.pt"
)

model.eval()

mobile_model = optimize_for_mobile(model)

mobile_model._save_for_lite_interpreter(
    "/content/drive/MyDrive/TinyBayes/run_2000/mobilenet_finetuned_576.ptl"
)

print("Saved PTL")

In [ ]:
import torch

m = torch.jit.load(
    "/content/drive/MyDrive/TinyBayes/run_2000/mobilenet_finetuned_576.ptl"
)

print(type(m))